# State-level agricultural GHG emissions in India: walkthrough

This notebook reads the panel and the saved results and reproduces the tables and figures.
It does **not** retrain models by default (the full validation takes roughly 10-15 minutes); set `RUN_SUITE = True`
in the first code cell to regenerate every result. Method and caveats: see `README.md`.

In [ ]:
import os, sys, subprocess
import numpy as np
import pandas as pd
try:
    from IPython.display import display, Image
except ImportError:          # allows running this notebook's code as a plain script
    display, Image = print, None

RUN_SUITE = False            # True: rebuild target + rerun validation (slow)

ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
RES = os.path.join(ROOT, "results", "state_panel")
FIG = os.path.join(ROOT, "results", "figures")
PY = sys.executable
if RUN_SUITE:
    for s in ["build_tier1_ghg_target", "reconcile_with_bur", "validation_suite", "bootstrap_ci", "shap_dynamic", "make_figures"]:
        subprocess.run([PY, os.path.join(ROOT, "scripts", s + ".py")], check=True, cwd=ROOT)
pd.set_option("display.width", 200)

## 1. The panel
37 states and union territories, 2004-2024, built from the RBI Handbook of Statistics on Indian States, the 2012 and 2019
Livestock Censuses, the CEA General Review and NASA POWER.

In [ ]:
panel = pd.read_csv(os.path.join(ROOT, "data/processed/state_panel/state_year_panel_with_target.csv"))
print(f"{panel.shape[0]} state-years, {panel.State.nunique()} states, "
      f"{panel.Year_Standardized.min()}-{panel.Year_Standardized.max()}, no duplicate keys: "
      f"{not panel.duplicated(['State', 'Year_Standardized']).any()}")
cover = panel[["Rice_Area", "Gross_Sown_Area", "Fertiliser_Consumption_NPK", "Annual_Rainfall_mm",
               "Temperature_C", "Stage_of_Extraction_pct"]].notna().mean().round(3)
display(cover.rename("share of state-years populated").to_frame())

## 2. The target and its reconciliation with India's inventory
The target is a bottom-up IPCC Tier-1 estimate (rice CH4, residue burning, livestock, fertiliser soil N2O). Grid
electricity is **not** included (Energy sector in national inventories). The 2020 national sum is compared with BUR4.

In [ ]:
comps = [c for c in panel.columns if c.startswith("GHG_")]
share = (panel[comps].sum() / panel[comps].sum().sum()).round(3).rename("share of target, all years")
display(share.to_frame())
display(pd.read_csv(os.path.join(RES, "bur_reconciliation.csv")).round(2))

## 3. Spatial generalisation (leave-one-state-out)
Feature selection is fitted inside each training fold; the target's own lags are never features. **Scale only** is a
regression of log emissions on log gross sown area.

In [ ]:
v = pd.read_csv(os.path.join(RES, "validation_suite.csv"))
spatial = v[v.scheme == "LOSO"].pivot_table(index="target", columns="model", values=["R2", "R2_log"])
display(spatial.round(3))

In [ ]:
b = pd.read_csv(os.path.join(RES, "bootstrap_r2_differences.csv"))
display(b[b.scheme == "LOSO"].round(3))
if Image is not None:
    display(Image(os.path.join(FIG, "fig1_spatial_vs_baselines.png")))

## 4. Where the models fail
Mean absolute percentage error of XGBoost for each held-out state against state size.

In [ ]:
ps = pd.read_csv(os.path.join(RES, "per_state_error_level_xgb.csv")).sort_values("mean_actual", ascending=False)
display(ps.round(0).head(5)); display(ps.round(0).tail(5))
if Image is not None:
    display(Image(os.path.join(FIG, "fig3_per_state_error.png")))

## 5. Temporal generalisation (expanding window)

In [ ]:
temporal = v[v.scheme == "Expanding"].pivot_table(index="target", columns="model", values="R2")
display(temporal.round(3))
display(b[b.scheme == "Expanding"].round(3))
if Image is not None:
    display(Image(os.path.join(FIG, "fig2_temporal_vs_baselines.png")))

## 6. What the dynamic-only model uses (SHAP, interpretation only)

In [ ]:
display(pd.read_csv(os.path.join(RES, "shap_dynamic_v2.csv")).head(8).round(1))

## Reading the results
* For the emission level of an unseen state, a one-variable size regression matches the machine-learning models
  (XGBoost minus scale-only R2 = -0.04, 95% CI -0.30 to 0.52); in log terms it is clearly better.
* On the dynamic-only target the models lead in raw R2 but the interval is far too wide to claim an advantage.
* No model beats persistence over time, and year-over-year change is not predictable.
* Errors for the smallest states are large (41% to over 1,200%).